# LangChain Root Listener Tracers Reference

Developer-facing statements defined in `langchain_core.tracers.root_listeners`.

# `Listener`

Synchronous listener accepted by `RootListenersTracer`.

```python
Listener = Callable[[Run], None] | Callable[[Run, RunnableConfig], None]
```

A listener can accept either the root `Run` alone or the root `Run` together with the tracer's `RunnableConfig`.

---

# `AsyncListener`

Asynchronous listener accepted by `AsyncRootListenersTracer`.

```python
AsyncListener = (
    Callable[[Run], Awaitable[None]]
    | Callable[[Run, RunnableConfig], Awaitable[None]]
)
```

A listener can accept either the root `Run` alone or the root `Run` together with the tracer's `RunnableConfig`.

---

# `RootListenersTracer: BaseTracer`

Synchronous tracer that invokes listeners when the root run starts, finishes successfully, or finishes with an error.

## Fields

```python
log_missing_parent = False # Whether to log a warning when a parent run is missing
config: RunnableConfig # Runnable configuration supplied to listeners
root_id: UUID | None = None # ID of the first run observed by the tracer
```

## Constructor

```python
RootListenersTracer(
    *,
    config: RunnableConfig, # Runnable configuration supplied to listeners
    on_start: Listener | None, # Listener invoked when the root run starts
    on_end: Listener | None, # Listener invoked when the root run succeeds
    on_error: Listener | None, # Listener invoked when the root run fails
) -> None
```

## Behaviour

The first run created through the inherited tracer callbacks becomes the root run and sets `root_id`. Later run creations do not invoke `on_start`.

When the root run is updated, `on_end` is invoked if `run.error` is `None`; otherwise, `on_error` is invoked. Updates for child runs are ignored.

Each configured listener is called with either `(run)` or `(run, config)`, according to the listener's accepted parameters.

Completed runs are not persisted by this tracer; its persistence hook is a no-op.

---

# `AsyncRootListenersTracer: AsyncBaseTracer`

Asynchronous tracer that awaits listeners when the root run starts, finishes successfully, or finishes with an error.

## Fields

```python
log_missing_parent = False # Whether to log a warning when a parent run is missing
config: RunnableConfig # Runnable configuration supplied to listeners
root_id: UUID | None = None # ID of the first run observed by the tracer
```

## Constructor

```python
AsyncRootListenersTracer(
    *,
    config: RunnableConfig, # Runnable configuration supplied to listeners
    on_start: AsyncListener | None, # Listener awaited when the root run starts
    on_end: AsyncListener | None, # Listener awaited when the root run succeeds
    on_error: AsyncListener | None, # Listener awaited when the root run fails
) -> None
```

## Behaviour

The first run created through the inherited asynchronous tracer callbacks becomes the root run and sets `root_id`. Later run creations do not invoke `on_start`.

When the root run is updated, `on_end` is awaited if `run.error` is `None`; otherwise, `on_error` is awaited. Updates for child runs are ignored.

Each configured listener is called with either `(run)` or `(run, config)`, according to the listener's accepted parameters.

Completed runs are not persisted by this tracer; its asynchronous persistence hook is a no-op.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import the real runnable class
from langchain_core.runnables.config import RunnableConfig # Import the runnable configuration type
from langchain_core.tracers.schemas import Run # Import the traced Run model


def add_two(number: int) -> int: # Define the first pipeline operation
    return number + 2 # Add two to the input


def multiply_by_three(number: int) -> int: # Define the second pipeline operation
    return number * 3 # Multiply the input by three


def on_start(run: Run, config: RunnableConfig) -> None: # Handle root-run creation
    print("\nSYNC ROOT STARTED") # Display a heading
    print("Name:", run.name) # Display the root run name
    print("Type:", run.run_type) # Display the root run type
    print("Inputs:", run.inputs) # Display the root run inputs
    print("Tags:", config.get("tags")) # Display tags from the runnable config


def on_end(run: Run, config: RunnableConfig) -> None: # Handle successful root completion
    print("\nSYNC ROOT COMPLETED") # Display a heading
    print("Name:", run.name) # Display the root run name
    print("Outputs:", run.outputs) # Display the root run outputs
    print("Error:", run.error) # Display None because the run succeeded


def on_error(run: Run, config: RunnableConfig) -> None: # Handle root-run failure
    print("\nSYNC ROOT FAILED") # Display a heading
    print("Name:", run.name) # Display the root run name
    print("Error:", run.error) # Display the recorded error


step1 = RunnableLambda(add_two).with_config(run_name="add_two") # Create the first child runnable
step2 = RunnableLambda(multiply_by_three).with_config(run_name="multiply_by_three") # Create the second child runnable

pipeline = (step1 | step2).with_listeners( # Attach listeners to the root pipeline
    on_start=on_start, # Register the root start listener
    on_end=on_end, # Register the root success listener
    on_error=on_error, # Register the root error listener
) # Finish configuring the pipeline

sync_result = pipeline.invoke( # Run the pipeline synchronously
    5, # Provide the pipeline input
    config={ # Configure this execution
        "run_name": "math_pipeline", # Set the root run name
        "tags": ["sync", "demo"], # Add tags to the root run
        "metadata": {"source": "jupyter"}, # Add metadata to the root run
    },
) # Finish invoking the pipeline

print("\nSynchronous result:", sync_result) # Display (5 + 2) * 3 = 21


async def subtract_one(number: int) -> int: # Define an asynchronous operation
    return number - 1 # Subtract one from the input


async def async_on_start(run: Run, config: RunnableConfig) -> None: # Handle async root creation
    print("\nASYNC ROOT STARTED") # Display a heading
    print("Name:", run.name) # Display the root run name
    print("Inputs:", run.inputs) # Display the root run inputs


async def async_on_end(run: Run, config: RunnableConfig) -> None: # Handle async root success
    print("\nASYNC ROOT COMPLETED") # Display a heading
    print("Name:", run.name) # Display the root run name
    print("Outputs:", run.outputs) # Display the root run outputs


async def async_on_error(run: Run, config: RunnableConfig) -> None: # Handle async root failure
    print("\nASYNC ROOT FAILED") # Display a heading
    print("Error:", run.error) # Display the recorded error


async_runnable = RunnableLambda(subtract_one).with_alisteners( # Attach real async listeners
    on_start=async_on_start, # Register the async start listener
    on_end=async_on_end, # Register the async success listener
    on_error=async_on_error, # Register the async error listener
) # Finish configuring the async runnable

async_result = await async_runnable.ainvoke( # Invoke directly inside Jupyter
    10, # Provide the input
    config={"run_name": "subtract_operation"}, # Name the root run
) # Finish invoking the runnable

print("\nAsynchronous result:", async_result) # Display 9